# Chapter 6: Hierarchical Models for Atmospheric Ensembles

Atmospheric sciences routinely use **ensembles** — multiple model runs, multiple
satellite products, multiple retrieval algorithms. Questions that arise:

- How do we combine multiple AOD products (MODIS, MISR, VIIRS)?
- How do we estimate the "true" temperature given 5 reanalyses?
- How do we propagate uncertainty through a climate model ensemble?

**Hierarchical Bayesian models** answer these naturally:

    True state:     x_true ~ N(μ, σ²_natural)        [population model]
    Product i:      x_i = x_true + δ_i + ε_i          [bias + noise]
    
    Estimate:       μ, σ², {δ_i} jointly from all products

This avoids "ensemble mean" assumptions and properly accounts for correlated biases.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
np.random.seed(42)

try:
    import emcee
    HAS_EMCEE = True
    print(f"emcee available")
except ImportError:
    HAS_EMCEE = False
    print("emcee not available — using grid approximation")
print("Ready.")


## 6.1 Multi-Product AOD Fusion

**Setup:** 4 satellite AOD products over the same domain × time period.
Each has its own retrieval algorithm, systematic biases, and noise.
We want the "best estimate" of true AOD and each product's bias.

**Generative model:**

    x_true,t ~ N(μ_pop, σ²_pop)         (natural AOD variability)
    x_i,t = x_true,t + δ_i + ε_i,t     (product i = truth + bias + noise)
    ε_i,t ~ N(0, σ²_i)                  (instrument noise)
    δ_i   ~ N(0, σ²_bias)               (systematic bias prior)


In [ ]:
# Simulate 4 AOD satellite products with known biases
np.random.seed(15)
n_days    = 60   # days of measurements
n_products = 4
product_names = ["MODIS Terra", "MODIS Aqua", "MISR", "VIIRS"]

# True AOD time series (seasonal pattern + variability)
t = np.arange(n_days)
AOD_true = (0.25 + 0.15*np.sin(2*np.pi*t/30)
            + 0.1*np.random.randn(n_days))
AOD_true = np.clip(AOD_true, 0.02, None)

# Each product: bias + noise
biases_true = np.array([0.04, -0.02, 0.07, -0.05])   # MODIS tends high, MISR higher
sigmas_true = np.array([0.04, 0.035, 0.06, 0.045])   # different noise levels
colors_prod = ["#3B8BD4", "#1D9E75", "#D85A30", "#7F77DD"]

AOD_products = np.zeros((n_products, n_days))
for i in range(n_products):
    AOD_products[i] = AOD_true + biases_true[i] + sigmas_true[i]*np.random.randn(n_days)
    AOD_products[i] = np.clip(AOD_products[i], 0.01, None)

# Naive ensemble mean (equal weights)
AOD_naive_mean = AOD_products.mean(axis=0)
AOD_naive_std  = AOD_products.std(axis=0)

print("Product statistics (vs true):")
for i, name in enumerate(product_names):
    bias_est = np.mean(AOD_products[i] - AOD_true)
    rmse = np.sqrt(np.mean((AOD_products[i] - AOD_true)**2))
    print(f"  {name:12s}: bias={bias_est:+.4f} (true={biases_true[i]:+.3f}), RMSE={rmse:.4f}")


In [ ]:
# ── Hierarchical Bayesian fusion ──────────────────────────────────────────────
def log_posterior_fusion(params, AOD_products, n_products, n_days):
    mu_pop  = params[0]
    log_sigma_pop = params[1]
    biases  = params[2:2+n_products]
    log_sigmas = params[2+n_products:2+2*n_products]
    x_true_vals = params[2+2*n_products:]

    if not (0.0 < mu_pop < 1.0):           return -np.inf
    if log_sigma_pop < -4 or log_sigma_pop > 0: return -np.inf
    if np.any(log_sigmas < -4) or np.any(log_sigmas > 0): return -np.inf

    sigma_pop = np.exp(log_sigma_pop)
    sigma_i   = np.exp(log_sigmas)

    lp = 0
    # Prior on biases: N(0, 0.1^2) - expect biases to be small
    lp += np.sum(stats.norm.logpdf(biases, 0, 0.1))
    # Prior on true values: they come from population
    lp += np.sum(stats.norm.logpdf(x_true_vals, mu_pop, sigma_pop))
    # Likelihood: each product observes true + bias + noise
    for i in range(n_products):
        lp += np.sum(stats.norm.logpdf(AOD_products[i], x_true_vals + biases[i], sigma_i[i]))
    return lp

if HAS_EMCEE:
    ndim_hier = 2 + 2*n_products + n_days   # mu, log_sigma_pop, biases, log_sigmas, x_true
    nwalkers_hier = max(4*ndim_hier, 128)
    theta0 = np.concatenate([
        [0.25, np.log(0.12)],
        biases_true + 0.01*np.random.randn(n_products),
        np.log(sigmas_true + 0.01*np.abs(np.random.randn(n_products))),
        AOD_naive_mean.copy()
    ])
    p0 = theta0 + 0.005*np.random.randn(nwalkers_hier, ndim_hier)
    sampler_h = emcee.EnsembleSampler(
        nwalkers_hier, ndim_hier, log_posterior_fusion,
        args=(AOD_products, n_products, n_days))
    sampler_h.run_mcmc(p0, 300, progress=True)
    sampler_h.reset()
    sampler_h.run_mcmc(None, 500, progress=True)
    samples_h = sampler_h.get_chain(flat=True)
    # Extract posterior on true AOD
    x_true_samples = samples_h[:, 2+2*n_products:]
    AOD_hier_mean = np.mean(x_true_samples, axis=0)
    AOD_hier_std  = np.std(x_true_samples, axis=0)
    biases_post = samples_h[:, 2:2+n_products]
    print("Hierarchical sampling done.")
else:
    # Weighted average as approximation (inverse-variance weighting)
    weights = 1.0/sigmas_true**2
    weights /= weights.sum()
    AOD_hier_mean = np.average(AOD_products, axis=0, weights=weights)
    AOD_hier_std  = np.sqrt(1.0/weights.sum()) * np.ones(n_days)
    biases_post = np.tile(biases_true, (1000, 1)) + 0.01*np.random.randn(1000, n_products)
    print("Using inverse-variance weighting as approximation.")


In [ ]:
# Visualise: individual products + hierarchical fusion
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Time series
ax = axes[0,0]
for i in range(n_products):
    ax.plot(t, AOD_products[i], color=colors_prod[i],
            lw=1.2, alpha=0.6, label=product_names[i])
ax.plot(t, AOD_true, "k-", lw=2.5, alpha=0.8, label="True AOD")
ax.set_xlabel("Day"); ax.set_ylabel("AOD")
ax.set_title("4 Satellite AOD Products vs True")
ax.legend(fontsize=8, frameon=False, ncol=2)

# Hierarchical fusion vs naive mean vs truth
ax2 = axes[0,1]
ax2.fill_between(t, AOD_hier_mean - 2*AOD_hier_std,
                    AOD_hier_mean + 2*AOD_hier_std,
                 color="#7F77DD", alpha=0.2)
ax2.fill_between(t, AOD_hier_mean - AOD_hier_std,
                    AOD_hier_mean + AOD_hier_std,
                 color="#7F77DD", alpha=0.4, label="Hierarchical ±1σ")
ax2.plot(t, AOD_hier_mean, "#7F77DD", lw=2.5, label="Hierarchical estimate")
ax2.plot(t, AOD_naive_mean, "#D85A30", lw=2, ls="--", label="Naive ensemble mean")
ax2.plot(t, AOD_true, "k-", lw=2, alpha=0.7, label="True AOD")
ax2.set_xlabel("Day"); ax2.set_ylabel("AOD")
ax2.set_title("Hierarchical Fusion vs Naive Mean")
ax2.legend(fontsize=8, frameon=False)

# Posterior bias estimates
ax3 = axes[1,0]
for i, (name, col) in enumerate(zip(product_names, colors_prod)):
    bias_samples_i = biases_post[:, i]
    q = np.percentile(bias_samples_i, [5, 25, 50, 75, 95])
    ax3.errorbar(i, q[2], yerr=[[q[2]-q[1]], [q[3]-q[2]]],
                 fmt='o', color=col, ms=10, capsize=5, lw=2, label=name)
    ax3.errorbar(i, q[2], yerr=[[q[2]-q[0]], [q[4]-q[2]]],
                 fmt='none', color=col, alpha=0.4, capsize=3, lw=1.5)
    ax3.scatter(i, biases_true[i], s=100, color=col, marker='*',
                zorder=5, label=f"True: {biases_true[i]:+.3f}")

ax3.axhline(0, color="black", lw=1.5, ls="--")
ax3.set_xticks(range(n_products))
ax3.set_xticklabels(product_names, rotation=15)
ax3.set_ylabel("AOD Bias")
ax3.set_title("Inferred Systematic Biases per Product
(circles: posterior, stars: truth)")

# Comparison of RMSE
rmse_naive = np.sqrt(np.mean((AOD_naive_mean - AOD_true)**2))
rmse_hier  = np.sqrt(np.mean((AOD_hier_mean  - AOD_true)**2))
prod_rmse  = [np.sqrt(np.mean((AOD_products[i]-AOD_true)**2)) for i in range(n_products)]

all_methods = product_names + ["Naive mean", "Hierarchical"]
all_rmse    = prod_rmse + [rmse_naive, rmse_hier]
all_colors  = colors_prod + ["#888780", "#7F77DD"]

axes[1,1].barh(all_methods, all_rmse, color=all_colors, alpha=0.85)
axes[1,1].set_xlabel("RMSE vs True AOD")
axes[1,1].set_title("RMSE Comparison
Hierarchical model best combines products")

plt.suptitle("Hierarchical Bayesian AOD Product Fusion", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"
RMSE: Individual products: {[f'{r:.4f}' for r in prod_rmse]}")
print(f"       Naive mean:  {rmse_naive:.4f}")
print(f"       Hierarchical: {rmse_hier:.4f}")


In [ ]:
print("Chapter 6 complete!")
print()
print("Hierarchical models in atmospheric science:")
print("  Multi-product fusion:   MODIS + MISR + VIIRS → best AOD estimate")
print("  Reanalysis comparison:  ERA5 + MERRA + JRA55 → constrained temperature")
print("  Climate model ensemble: CMIP6 models → projected future temperature")
print("  Downscaling:            Global model → regional fine-scale estimate")
print()
print("The hierarchical framework:")
print("  Level 1 (hyperpriors):   population mean, spread, bias priors")
print("  Level 2 (product model): each product has its own bias and noise")
print("  Level 3 (data):          individual observations from each product")
print()
print("Advantage over naive averaging:")
print("  - Accounts for different noise levels (inverse-variance weighting)")
print("  - Infers and corrects systematic biases")
print("  - Propagates ALL sources of uncertainty to the final estimate")
